#Paquetes necesarios

In [4]:
#%pip install opencv-python ultralytics lap

In [5]:
import cv2  
import math 

from ultralytics import YOLO

In [6]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("CUDA de PyTorch:", torch.version.cuda)
print("Número de GPU:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: c:\Users\Usuario\miniconda3\envs\VC_PRACTICA_4\python.exe
PyTorch: 2.6.0+cu124
CUDA disponible: True
CUDA de PyTorch: 12.4
Número de GPU: 1
GPU: NVIDIA GeForce GTX 1050


In [7]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA de PyTorch:", torch.version.cuda)

PyTorch: 2.6.0+cu124
CUDA disponible: True
GPU: NVIDIA GeForce GTX 1050
CUDA de PyTorch: 12.4


Modelos preentrenados, visualizando con las utilidades de ultralytics

In [8]:
# Carga del modelo
#model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolo11n-seg.pt') #Máscaras
model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model(filename, show=True)

cv2.destroyAllWindows()


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) c:\Users\Usuario\Desktop\VC\VC\PRACTICA_4\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 20.2ms
video 1/1 (frame 2/375) c:\Users\Usuario\Desktop\VC\VC\PRACTICA_4\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 20.1ms
video 1/1 (frame 3/375) c:\Users\Usuario\Desktop\VC\VC\PRACTICA_4\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 20.4ms
video 1/1 (frame 4/375) c:\Users\Usuario\Desktop\VC\VC\PRACTICA_4\P4\TGC23_PdH_C0056cut

Desde cámara, detección con yolo11, modelo nano. Visualización propia con OpenCV

In [9]:
# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') #Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck", "boat",
              "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
              "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
              "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat",
              "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
              "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
              "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa", "pottedplant", "bed",
              "diningtable", "toilet", "tvmonitor", "laptop", "mouse", "remote", "keyboard", "cell phone",
              "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
              "teddy bear", "hair drier", "toothbrush"
              ]


# Captura desde la webcam
vid = cv2.VideoCapture(0)
  
while(True):      
    # fotograma a fotograma
    ret, img = vid.read()
  
    # si hay imagen válida
    if ret:  
        # Detecta en la imagen
        results = model(img, stream=True)
        
        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) # convert to int values
                
                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza --->",confidence)

                # Clase
                cls = int(box.cls[0])
                print("Clase -->", classNames[cls])

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)
                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2
                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

        # Muestra fotograma
        cv2.imshow('Vid', img)
    
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Destruye ventanas
cv2.destroyAllWindows()


0: 480x640 1 person, 2 chairs, 12.4ms
Confianza ---> 0.94
Clase --> person
Confianza ---> 0.58
Clase --> chair
Confianza ---> 0.37
Clase --> chair
Speed: 1.6ms preprocess, 12.4ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 2 chairs, 10.6ms
Confianza ---> 0.93
Clase --> person
Confianza ---> 0.64
Clase --> chair
Confianza ---> 0.29
Clase --> chair
Speed: 1.6ms preprocess, 10.6ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 2 chairs, 10.5ms
Confianza ---> 0.93
Clase --> person
Confianza ---> 0.51
Clase --> chair
Confianza ---> 0.3
Clase --> chair
Speed: 1.4ms preprocess, 10.5ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 2 chairs, 10.0ms
Confianza ---> 0.93
Clase --> person
Confianza ---> 0.64
Clase --> chair
Confianza ---> 0.35
Clase --> chair
Speed: 2.8ms preprocess, 10.0ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person,

Seguimiento. Requiere instalar lap con pip install lap

In [10]:
from collections import defaultdict
import numpy as np

# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') #Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car"]


# Captura desde la webcam
vid = cv2.VideoCapture(0)
track_history = defaultdict(lambda: [])
  
while(True):      
    # fotograma a fotograma
    ret, img = vid.read()
  
    # si hay imagen válida
    if ret:  
        # Seguimiento, con persistencia entre fotogramas
        results = model.track(img, persist=True, classes = [0,1,2])

        if 0:
            if results is not None:
                print(results[0])
                boxes = results[0].boxes.xywh.cpu()
                track_ids = results[0].boxes.id.int().cpu().tolist()
                annotated_frame = results[0].plot()
                for box, track_id in zip(boxes, track_ids):
                    x, y, w, h = box
                    track = track_history[track_id]
                    track.append((float(x), float(y)))
                    if len(track) > 30:
                        track.pop(0)
                    points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                    cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)
                cv2.imshow("YOLO11 Tracking", annotated_frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
        

        
        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) # convert to int values

                #Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                else:
                    track_id = ''
                
                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza --->",confidence)

                # Clase
                cls = int(box.cls[0])
                print("Clase -->", classNames[cls])

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)
                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2
                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

        # Muestra fotograma
        cv2.imshow('Vid', img)
    
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Destruye ventanas
cv2.destroyAllWindows()




0: 480x640 1 person, 14.6ms
Speed: 1.8ms preprocess, 14.6ms inference, 18.4ms postprocess per image at shape (1, 3, 480, 640)
Confianza ---> 0.94
Clase --> person

0: 480x640 1 person, 11.3ms
Speed: 1.2ms preprocess, 11.3ms inference, 3.6ms postprocess per image at shape (1, 3, 480, 640)
Confianza ---> 0.94
Clase --> person

0: 480x640 1 person, 11.9ms
Speed: 2.8ms preprocess, 11.9ms inference, 3.4ms postprocess per image at shape (1, 3, 480, 640)
Confianza ---> 0.93
Clase --> person

0: 480x640 1 person, 11.5ms
Speed: 1.1ms preprocess, 11.5ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
Confianza ---> 0.93
Clase --> person

0: 480x640 1 person, 12.3ms
Speed: 1.2ms preprocess, 12.3ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)
Confianza ---> 0.92
Clase --> person

0: 480x640 1 person, 11.5ms
Speed: 1.2ms preprocess, 11.5ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)
Confianza ---> 0.91
Clase --> person

0: 480x640 1 person,

Intregración con seguimiento (tracking)
!!!!!!!!!Nota: he tenido que bajar a la versión de python 3.9.5 e instalar lap con pip install lap

In [11]:
# Carga del modelo
model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolov11n-seg.pt') #Máscaras
#model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model.track(source=filename, show=True)  # BoT-SORT tracker (por defecto)
#results = model.track(source=filename, show=True, tracker="bytetrack.yaml")  # ByteTrack tracker

cv2.destroyAllWindows()


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) c:\Users\Usuario\Desktop\VC\VC\PRACTICA_4\P4\TGC23_PdH_C0056cut.mp4: 384x640 4 persons, 1 bird, 49.2ms
video 1/1 (frame 2/375) c:\Users\Usuario\Desktop\VC\VC\PRACTICA_4\P4\TGC23_PdH_C0056cut.mp4: 384x640 4 persons, 26.8ms
video 1/1 (frame 3/375) c:\Users\Usuario\Desktop\VC\VC\PRACTICA_4\P4\TGC23_PdH_C0056cut.mp4: 384x640 4 persons, 1 bird, 27.7ms
video 1/1 (frame 4/375) c:\Users\Usuario\Desktop\VC\VC\PRACTICA_4\P4\TGC23_PdH_C0056cut.m